# Member 2 — Encoding Categorical Variables

**Technique:** Convert categorical fields (`label`, `region`, `folder`) into numeric codes ML models can consume.

## Why this dataset needs it
Folder names carry the **class** (Healthy / Unhealthy) and **region** (Amravati / Nagpur / Pune / Unknown). Models and correlation plots need integers or one-hot columns — not free text.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve Group_Deliverable root whether cwd is notebooks/ or deliverable root
HERE = Path.cwd().resolve()
ROOT = None
for p in (HERE, *HERE.parents):
    if (p / "src" / "preprocess_utils.py").exists():
        ROOT = p
        break
    if (p / "Group_Deliverable" / "src" / "preprocess_utils.py").exists():
        ROOT = p / "Group_Deliverable"
        break
if ROOT is None:
    raise FileNotFoundError("Run from progress/Group_Deliverable or its notebooks/ folder.")

sys.path.insert(0, str(ROOT / "src"))
from preprocess_utils import (
    SEED, SOURCE_URL, DOI, FOLDERS, CLASSES, paths,
    inventory_table, discover_images, audit_images,
    remove_exact_duplicates, iqr_mask, extract_feature_matrix, read_rgb,
    stratified_sample,
)

P = paths(ROOT)
RAW, VIZ, OUT, LOGS = P["raw"], P["viz"], P["outputs"], P["logs"]
for d in (VIZ, OUT, LOGS):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(SEED)
print("Deliverable root:", ROOT)
print("Raw data:", RAW)
print("Dataset:", SOURCE_URL, "| DOI:", DOI)


## 1. Load audited index (or rebuild) and encode categories


In [ ]:
index_path = OUT / "m1_valid_image_index.csv"
if index_path.exists():
    df = pd.read_csv(index_path)
    print("Loaded Member 1 valid index:", len(df))
else:
    df, rejected = audit_images(RAW)
    print("Audited fresh valid images:", len(df), "| rejected:", len(rejected))

# Label encoding (ordered class list for reproducibility)
label_map = {c: i for i, c in enumerate(CLASSES)}
df["label_encoded"] = df["label"].map(label_map).astype(int)

# Region encoding
regions = sorted(df["region"].dropna().unique().tolist())
region_map = {r: i for i, r in enumerate(regions)}
df["region_encoded"] = df["region"].map(region_map).astype(int)

# One-hot for region (useful for linear models / EDA)
region_dummies = pd.get_dummies(df["region"], prefix="region")
encoded = pd.concat([df, region_dummies], axis=1)

display(pd.DataFrame({"category": ["label", "region"], "mapping": [label_map, region_map]}))
display(encoded[["path", "label", "label_encoded", "region", "region_encoded"]].head(10))

encoded.to_csv(OUT / "m2_encoded_metadata.csv", index=False)
(LOGS / "m2_encoding_maps.json").write_text(
    json.dumps({"label_map": label_map, "region_map": region_map}, indent=2),
    encoding="utf-8",
)


## 2. EDA visualization — encoded category distributions


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label codes
lc = encoded["label_encoded"].value_counts().sort_index()
axes[0].bar([CLASSES[i] for i in lc.index], lc.values, color=["#2ca02c", "#d62728"])
axes[0].set_title("Encoded labels (0=Healthy, 1=Unhealthy)")
axes[0].set_ylabel("Count")

# Region codes
rc = encoded.groupby("region")["region_encoded"].count().sort_values(ascending=False)
axes[1].bar(rc.index, rc.values, color="#9467bd")
axes[1].set_title("Images per region category")
axes[1].tick_params(axis="x", rotation=20)
axes[1].set_ylabel("Count")

fig.tight_layout()
fig.savefig(VIZ / "m2_categorical_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Interpretation: encoding preserves class/region structure while making columns numeric for downstream scaling and modelling.")


## Viva talking points
1. Contrast label encoding vs one-hot encoding.
2. Justify mapping Healthy→0, Unhealthy→1 from folder names.
3. Interpret the region distribution chart (class imbalance by capture site).
